In [ ]:
# KLM Fig. 2(a): standalone CPU/NumPy notebook.
# This is a CPU port of the bridge/streaming structure used by the JAX notebook.
# Full h_ref=2^-25 is the default; use FIG2A_RUN_MODE=smoke for
# a quick integrity check. Use kaggle_klm_fig2a_JAX.ipynb on a P100 for the paper-scale run.

from __future__ import annotations

import csv
import os
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

RUN_MODE = os.environ.get("FIG2A_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("FIG2A_RUN_MODE must be 'full' or 'smoke'")

KAPPA = 2.0
LAMBDA = 0.05
SIGMA = 0.2
FINAL_TIME = 1.0
INITIAL_Y = 0.02
INITIAL_X = INITIAL_Y**2
ALPHA = (4.0 * KAPPA * LAMBDA - SIGMA**2) / 8.0
BETA = -KAPPA / 2.0
GAMMA = SIGMA / 2.0
LEVELS = np.array([4, 5, 6, 7, 8, 9], dtype=int)
HMAX_BY_LEVEL = 2.0 ** (-LEVELS.astype(float))
RHO = 64.0
SEED = int(os.environ.get("FIG2A_SEED", "2022"))

if RUN_MODE == "smoke":
    NUMBER_OF_PATHS = int(os.environ.get("FIG2A_M", "32"))
    REFERENCE_POWER = int(os.environ.get("FIG2A_REFERENCE_POWER", "12"))
    FINE_STEPS_PER_CHUNK = int(os.environ.get("FIG2A_CHUNK_STEPS", "512"))
else:
    NUMBER_OF_PATHS = int(os.environ.get("FIG2A_M", "1000"))
    REFERENCE_POWER = int(os.environ.get("FIG2A_REFERENCE_POWER", "25"))
    FINE_STEPS_PER_CHUNK = int(os.environ.get("FIG2A_CHUNK_STEPS", str(2**14)))

REFERENCE_STEP = 2.0 ** (-REFERENCE_POWER)
NUMBER_OF_FINE_STEPS = 2**REFERENCE_POWER
FINE_STEPS_PER_CHUNK = min(FINE_STEPS_PER_CHUNK, NUMBER_OF_FINE_STEPS)
if NUMBER_OF_FINE_STEPS % FINE_STEPS_PER_CHUNK != 0:
    raise ValueError("FIG2A_CHUNK_STEPS must divide 2^FIG2A_REFERENCE_POWER")
NUMBER_OF_CHUNKS = NUMBER_OF_FINE_STEPS // FINE_STEPS_PER_CHUNK

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "klm_fig2a_numpy_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def implicit_lamperti_step(y_old, brownian_increment, step_size):
    u = y_old + GAMMA * brownian_increment
    denominator = 1.0 - BETA * step_size
    return u / (2.0 * denominator) + np.sqrt(
        u * u / (4.0 * denominator * denominator) + ALPHA * step_size / denominator
    )


def choose_adaptive_step_size(y_values, current_times):
    proposed = HMAX_BY_LEVEL[:, None] * np.minimum(1.0, np.abs(y_values))
    minimum = (HMAX_BY_LEVEL / RHO)[:, None]
    used_minimum = proposed <= minimum
    chosen = np.where(used_minimum, minimum, proposed)
    return np.minimum(chosen, FINAL_TIME - current_times), used_minimum


def make_brownian_increment_chunk(chunk_number):
    rng = np.random.default_rng(SEED + 10_000 * chunk_number)
    return rng.standard_normal((FINE_STEPS_PER_CHUNK, NUMBER_OF_PATHS)) * np.sqrt(REFERENCE_STEP)


def bridge_value(w_left, w_right, t_left, t_right, target_time, rng):
    tau = np.clip(target_time, t_left, t_right)
    theta = (tau - t_left) / REFERENCE_STEP
    mean = w_left[None, :] + theta * (w_right - w_left)[None, :]
    var = np.maximum((tau - t_left) * (t_right - tau) / REFERENCE_STEP, 0.0)
    return mean + np.sqrt(var) * rng.standard_normal(target_time.shape)


def update_adaptive(y, last_w, last_t, target_t, used_minimum, n_steps, sum_steps, w_left, w_right, t_left, t_right, rng, use_sia):
    mask = target_t <= t_right + 1e-15
    if not np.any(mask):
        return y, last_w, last_t, target_t, used_minimum, n_steps, sum_steps
    target_w_raw = bridge_value(w_left, w_right, t_left, t_right, target_t, rng)
    target_w = np.where(mask, target_w_raw, last_w)
    h = target_t - last_t
    dW = target_w - last_w
    if use_sia:
        explicit = (y + h * ALPHA / y + GAMMA * dW) / (1.0 - BETA * h)
    else:
        explicit = y + h * (ALPHA / y + BETA * y) + GAMMA * dW
    backstop = implicit_lamperti_step(y, dW, h)
    stepped = np.where(used_minimum | (explicit <= 0.0), backstop, explicit)
    y = np.where(mask, stepped, y)
    last_w = np.where(mask, target_w, last_w)
    last_t = np.where(mask, target_t, last_t)
    n_steps = n_steps + mask.astype(int)
    sum_steps = sum_steps + np.where(mask, h, 0.0)
    next_h, next_used = choose_adaptive_step_size(y, last_t)
    target_t = np.where(mask, last_t + next_h, target_t)
    used_minimum = np.where(mask, next_used, used_minimum)
    return y, last_w, last_t, target_t, used_minimum, n_steps, sum_steps


def run_pass1():
    level_path_shape = (len(LEVELS), NUMBER_OF_PATHS)
    initial_h, initial_used = choose_adaptive_step_size(
        np.full(level_path_shape, INITIAL_Y), 0.0
    )
    reference_y = np.full(NUMBER_OF_PATHS, INITIAL_Y)
    brownian_w = np.zeros(NUMBER_OF_PATHS)
    ea_y = np.full(level_path_shape, INITIAL_Y)
    sia_y = np.full(level_path_shape, INITIAL_Y)
    ea_last_w = np.zeros(level_path_shape)
    sia_last_w = np.zeros(level_path_shape)
    ea_last_t = np.zeros(level_path_shape)
    sia_last_t = np.zeros(level_path_shape)
    ea_target_t = initial_h.copy()
    sia_target_t = initial_h.copy()
    ea_used = initial_used.copy()
    sia_used = initial_used.copy()
    ea_n = np.zeros(level_path_shape, dtype=int)
    sia_n = np.zeros(level_path_shape, dtype=int)
    ea_sum = np.zeros(level_path_shape)
    sia_sum = np.zeros(level_path_shape)
    rng_ea = np.random.default_rng(SEED + 100_000)
    rng_sia = np.random.default_rng(SEED + 200_000)
    start = time.perf_counter()
    for chunk_number in range(NUMBER_OF_CHUNKS):
        dW_chunk = make_brownian_increment_chunk(chunk_number)
        for local_index, dW in enumerate(dW_chunk):
            fine_index = chunk_number * FINE_STEPS_PER_CHUNK + local_index + 1
            t_right = fine_index * REFERENCE_STEP
            t_left = t_right - REFERENCE_STEP
            w_left = brownian_w
            w_right = w_left + dW
            reference_y = implicit_lamperti_step(reference_y, dW, REFERENCE_STEP)
            ea_y, ea_last_w, ea_last_t, ea_target_t, ea_used, ea_n, ea_sum = update_adaptive(
                ea_y, ea_last_w, ea_last_t, ea_target_t, ea_used, ea_n, ea_sum,
                w_left, w_right, t_left, t_right, rng_ea, use_sia=False,
            )
            sia_y, sia_last_w, sia_last_t, sia_target_t, sia_used, sia_n, sia_sum = update_adaptive(
                sia_y, sia_last_w, sia_last_t, sia_target_t, sia_used, sia_n, sia_sum,
                w_left, w_right, t_left, t_right, rng_sia, use_sia=True,
            )
            brownian_w = w_right
        if chunk_number % max(1, NUMBER_OF_CHUNKS // 8) == 0:
            print(f"pass 1 chunk {chunk_number + 1}/{NUMBER_OF_CHUNKS}")
    print(f"pass 1 completed in {time.perf_counter() - start:.2f}s")
    return dict(reference_y=reference_y, ea_y=ea_y, sia_y=sia_y, ea_n=ea_n, sia_n=sia_n, ea_sum=ea_sum, sia_sum=sia_sum)


def run_pass2(h_mean_by_level):
    level_path_shape = (len(LEVELS), NUMBER_OF_PATHS)
    brownian_w = np.zeros(NUMBER_OF_PATHS)
    ef_y = np.full(level_path_shape, INITIAL_Y)
    ef_x = np.full(level_path_shape, INITIAL_X)
    ft_raw = np.full(level_path_shape, INITIAL_X)
    fixed_if_y = np.full(level_path_shape, INITIAL_Y)
    last_w = np.zeros(level_path_shape)
    last_t = np.zeros(level_path_shape)
    fixed_step = h_mean_by_level.astype(float)
    target_t = np.broadcast_to(np.minimum(fixed_step[:, None], FINAL_TIME), level_path_shape).copy()
    rng_fixed = np.random.default_rng(SEED + 300_000)
    start = time.perf_counter()
    for chunk_number in range(NUMBER_OF_CHUNKS):
        dW_chunk = make_brownian_increment_chunk(chunk_number)
        for local_index, dW in enumerate(dW_chunk):
            fine_index = chunk_number * FINE_STEPS_PER_CHUNK + local_index + 1
            t_right = fine_index * REFERENCE_STEP
            t_left = t_right - REFERENCE_STEP
            w_left = brownian_w
            w_right = w_left + dW
            mask = target_t <= t_right + 1e-15
            if np.any(mask):
                target_w_raw = bridge_value(w_left, w_right, t_left, t_right, target_t, rng_fixed)
                target_w = np.where(mask, target_w_raw, last_w)
                h = target_t - last_t
                scheme_dW = target_w - last_w
                ef_y_candidate = ef_y + h * (ALPHA / ef_y + BETA * ef_y) + GAMMA * scheme_dW
                ef_x_candidate = ef_x + h * KAPPA * (LAMBDA - ef_x) + SIGMA * np.sqrt(np.abs(ef_x)) * scheme_dW
                positive_part = np.maximum(ft_raw, 0.0)
                ft_candidate = ft_raw + h * KAPPA * (LAMBDA - positive_part) + SIGMA * np.sqrt(positive_part) * scheme_dW
                if_candidate = implicit_lamperti_step(fixed_if_y, scheme_dW, h)
                ef_y = np.where(mask, ef_y_candidate, ef_y)
                ef_x = np.where(mask, ef_x_candidate, ef_x)
                ft_raw = np.where(mask, ft_candidate, ft_raw)
                fixed_if_y = np.where(mask, if_candidate, fixed_if_y)
                last_w = np.where(mask, target_w, last_w)
                last_t = np.where(mask, target_t, last_t)
                next_target = np.minimum(last_t + fixed_step[:, None], FINAL_TIME)
                target_t = np.where(mask, next_target, target_t)
            brownian_w = w_right
        if chunk_number % max(1, NUMBER_OF_CHUNKS // 8) == 0:
            print(f"pass 2 chunk {chunk_number + 1}/{NUMBER_OF_CHUNKS}")
    print(f"pass 2 completed in {time.perf_counter() - start:.2f}s")
    return dict(ef_y=ef_y, ef_x=ef_x, fixed_if_y=fixed_if_y, ft_raw=ft_raw)


def rmse(scheme_x, reference_x):
    diff = scheme_x - reference_x[None, :]
    return np.sqrt(np.mean(diff * diff, axis=1))


def fit_order(x, y):
    return float(np.polyfit(np.log(x), np.log(np.maximum(y, 1e-300)), 1)[0])


def plot_results(h_mean, errors):
    style = {
        "EA": dict(color="blue", marker="o", ls="-", mfc="none"),
        "EF_y": dict(color="black", marker="+", ls="-"),
        "EF_x": dict(color="0.45", marker="s", ls="--", mfc="none"),
        "IF": dict(color="magenta", marker="o", ls="-.", mfc="none"),
        "SIA": dict(color="cyan", marker="x", ls="--"),
        "FT": dict(color="red", marker="x", ls=":"),
    }
    fig, ax = plt.subplots(figsize=(5.8, 4.5))
    for name in ["EA", "EF_y", "EF_x", "IF", "SIA", "FT"]:
        order = fit_order(h_mean, errors[name])
        ax.loglog(h_mean, errors[name], label=f"{name} ({order:.2f})", **style[name])
    ax.loglog(h_mean, 0.55 * errors["EA"][-1] * (h_mean / h_mean[-1]), "--", color="tab:orange", lw=1)
    ax.loglog(h_mean, 2.7 * errors["EA"][-1] * (h_mean / h_mean[-1]) ** 0.5, "-", color="steelblue", lw=1)
    ax.set_xlabel("h_mean")
    ax.set_ylabel("RMSE")
    ax.set_title(f"KLM Fig. 2(a), NumPy bridge stream, h=2^-{REFERENCE_POWER}")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    stem = f"fig2a_bridge_streaming_numpy_h{REFERENCE_POWER}"
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=220)
    fig.savefig(OUT_DIR / f"{stem}.pdf")
    with (OUT_DIR / f"{stem}.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["level", "h_mean", "EA", "EF_y", "EF_x", "IF", "SIA", "FT"])
        for i, level in enumerate(LEVELS):
            writer.writerow([int(level), float(h_mean[i])] + [float(errors[k][i]) for k in ["EA", "EF_y", "EF_x", "IF", "SIA", "FT"]])
    print(f"saved {OUT_DIR / (stem + '.png')}")
    print(f"saved {OUT_DIR / (stem + '.csv')}")


def main():
    print(f"Standalone NumPy KLM Fig. 2(a), RUN_MODE={RUN_MODE}")
    print(f"M={NUMBER_OF_PATHS}, h_ref=2^-{REFERENCE_POWER}, fine steps={NUMBER_OF_FINE_STEPS:,}, chunks={NUMBER_OF_CHUNKS}")
    pass1 = run_pass1()
    reference_x = pass1["reference_y"] ** 2
    ea_x = pass1["ea_y"] ** 2
    sia_x = pass1["sia_y"] ** 2
    h_mean = np.mean(pass1["ea_sum"] / np.maximum(pass1["ea_n"], 1), axis=1)
    print("h_mean:", h_mean)
    pass2 = run_pass2(h_mean)
    errors = {
        "EA": rmse(ea_x, reference_x),
        "EF_y": rmse(pass2["ef_y"] ** 2, reference_x),
        "EF_x": rmse(pass2["ef_x"], reference_x),
        "IF": rmse(pass2["fixed_if_y"] ** 2, reference_x),
        "SIA": rmse(sia_x, reference_x),
        "FT": rmse(np.maximum(pass2["ft_raw"], 0.0), reference_x),
    }
    for i, level in enumerate(LEVELS):
        print(int(level), float(h_mean[i]), *(float(errors[k][i]) for k in ["EA", "EF_y", "EF_x", "IF", "SIA", "FT"]))
    plot_results(h_mean, errors)


main()
